# Notebook 1: Quick Start With The Sample Project

This notebook is the shortest path from repository clone to a useful signal. It starts by validating the local environment, inspects the bundled Java sample, and then moves into graph queries you can use after Neo4j has been populated.

Use the first half of the notebook to confirm the repository is wired correctly. Use the query sections once you have loaded a graph with classes, methods, dependencies, and related metadata.

In [ ]:
# Import required libraries
import sys
from pathlib import Path
import os
from dotenv import load_dotenv

# Load environment variables from the repository root
env_path = Path('..') / '.env'
load_dotenv(dotenv_path=env_path)

# Add src to path
sys.path.append(str(Path('../src').resolve()))

from neo4j_agent import Neo4jAgent

print(f"Using environment file: {env_path}")
print("✅ Imports successful")

## Step 1: Establish A Working Connection

The goal here is simple: confirm that the notebook can read local configuration and talk to Neo4j. If the database is not running yet, this is the point where you find out, before moving deeper into the workflow.

In [ ]:
# Get credentials from environment
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

# Initialize Neo4j agent
agent = Neo4jAgent(
    uri=NEO4J_URI,
    user=NEO4J_USER,
    password=NEO4J_PASSWORD
)

status = agent.verify_connection()
print(f"✅ Connected to Neo4j at {status['uri']}")
print(f"   Reachable: {status['connected']}")

## Step 2: Inspect The Sample Project Before Ingestion

The public repository currently exposes a preflight-style inspection flow. That means this step tells you what the project contains, which build dependencies are declared, and which Java packages are present before you invest in a heavier ingestion pipeline.

In [ ]:
# Define project paths
project_path = Path('../examples/sample_project')
build_xml_path = project_path / 'build.xml'

print("🔎 Inspecting sample project...")
report = agent.ingest_project(
    version='v1.0.0',
    project_path=project_path,
    build_xml_path=str(build_xml_path)
 )

snapshot = report['project_snapshot']

print("\n✅ Project Preflight Complete")
print(f"   Status: {report['status']}")
print(f"   Java files: {snapshot['java_file_count']}")
print(f"   Packages: {snapshot['package_count']}")
print(f"   Build dependencies: {snapshot['build_dependency_count']}")
print("\nSample files:")
for path in snapshot['sample_java_files']:
    print(f"  • {path}")
print("\nBuild dependencies:")
for dependency in snapshot['build_dependencies']:
    print(f"  • {dependency}")

## Step 3: Explore The Graph Once Data Is Loaded

The next cells assume your Neo4j database already contains graph data for classes, methods, imports, and dependencies. If you have only run the preflight step so far, treat this section as a reusable query reference for later.

In [ ]:
# Query 1: List all classes
print("📊 Query 1: All Classes\n")

query = """
MATCH (c:Class)
RETURN c.name AS ClassName, c.category AS Category, c.gitTag AS Version
ORDER BY c.name
"""

results = agent.query(query)
for record in results:
    print(f"  • {record['ClassName']} ({record['Category']}) - {record['Version']}")

In [ ]:
# Query 2: Find class methods
print("📊 Query 2: Methods in ExampleTestMethod\n")

query = """
MATCH (c:Class {name: 'ExampleTestMethod'})-[:DEFINES_METHOD]->(m:Method)
RETURN m.name AS MethodName, m.visibility AS Visibility, m.returnType AS ReturnType
ORDER BY m.name
"""

results = agent.query(query)
for record in results:
    print(f"  • {record['Visibility']} {record['ReturnType']} {record['MethodName']}()")

In [ ]:
# Query 3: Find inheritance hierarchy
print("📊 Query 3: Inheritance Hierarchy\n")

query = """
MATCH path = (child:Class)-[:EXTENDS]->(parent:Class)
RETURN child.name AS Child, parent.name AS Parent
"""

results = agent.query(query)
for record in results:
    print(f"  {record['Child']} → {record['Parent']}")

In [ ]:
# Query 4: Node statistics
print("📊 Query 4: Database Statistics\n")

query = """
MATCH (n)
RETURN labels(n)[0] AS NodeType, count(*) AS Count
ORDER BY Count DESC
"""

results = agent.query(query)
for record in results:
    print(f"  {record['NodeType']}: {record['Count']}")

## Step 4: Turn Query Results Into Something Readable

Raw query rows are useful, but tables make patterns easier to spot. This section converts graph output into a small dataframe so you can quickly inspect categories, coverage, and shape.

In [ ]:
import pandas as pd

# Get all classes with their categories
query = """
MATCH (c:Class)
RETURN c.name AS Class, c.category AS Category, c.visibility AS Visibility
ORDER BY c.category, c.name
"""

results = agent.query(query)
df = pd.DataFrame(results)

print("\n📊 Classes DataFrame:")
print(df)

print("\n📈 Category Distribution:")
print(df['Category'].value_counts())

## Step 5: Close Cleanly

Explicit cleanup matters in notebooks because it keeps reruns predictable. Closing the driver here avoids carrying an open connection across unrelated experiments.

In [ ]:
# Close Neo4j connection
agent.close()
print("✅ Neo4j connection closed")

## Where To Go Next

If this notebook gave you a clean preflight and a working connection, you are ready for deeper graph exploration. A sensible next move is to compare structural queries, inspect documentation coverage, and then connect those graph lookups to your MCP workflow.

Useful references:

- [README.md](../README.md) for the project overview.
- [GETTING_STARTED.md](../GETTING_STARTED.md) for setup and command-line usage.
- [ARCHITECTURE.md](../docs/ARCHITECTURE.md) for the repository design.
- [MCP_INTEGRATION.md](../docs/MCP_INTEGRATION.md) for editor integration.